# Joint GW + EM Fitting of ZTFJ1539+5027 with Real GBGPU Waveforms

This notebook demonstrates multi-messenger parameter estimation for a LISA verification galactic binary using:

- **Electromagnetic (EM)**: synthetic eclipsing light curves computed with `lcurve_rs`
- **Gravitational wave (GW)**: real LISA TDI response computed via `GBGPU.inject_signal()` + A/E channel noise PSDs

Shared parameter: **inclination** (`iangle_deg` in EM, `iota_rad` in GW), linked via a deg→rad transform.

We use the `JointFitter` from `lisatools.sampling.joint` and the eryn parallel-tempered sampler.

In [ ]:
import os, sys, tempfile, time
import numpy as np
import pandas as pd

sys.path.insert(0, '/fred/oz480/mcoughli/LISAanalysistools/src')
sys.path.insert(0, '/fred/oz480/mcoughli/GBGPU/src')

import lcurve_rs
from lcurve_rs.fitting import Fitter, FitResult, Prior
from lisatools.sampling.joint import JointFitter, ParameterMapping
from gbgpu.gbgpu import GBGPU
from lisatools.sensitivity import A1TDISens, E1TDISens
from eryn.prior import ProbDistContainer, uniform_dist

## 1. Load VGB catalogue and select target

We pick ZTFJ1539+5027 — a short-period, high-inclination system bright in both EM and GW.

In [ ]:
VGB_PATH = '/fred/oz480/mcoughli/LISAanalysistools/examples/vgbs.txt'
vgbs = pd.read_csv(VGB_PATH, index_col=0)
vgbs['Name'] = vgbs['Name'].str.strip("b'\"")

row = vgbs[vgbs['Name'] == 'ZTFJ1539'].iloc[0]

# GBGPU 9-parameter set
true_gw = {
    'amp': row['Amplitude'],
    'f0': row['Frequency'],
    'fdot': row['FrequencyDerivative'],
    'fddot': row['Drift'],
    'phi0': row['InitialPhase'],
    'iota': row['Inclination'],
    'psi': row['Polarization'],
    'lam': row['EclipticLongitude'],
    'beta': row['EclipticLatitude'],
}

# EM parameters
true_q = min(row['Mass1'], row['Mass2']) / max(row['Mass1'], row['Mass2'])
true_iangle_deg = np.degrees(row['Inclination'])

print('Target: ZTFJ1539+5027')
print(f'  GW amp       = {true_gw["amp"]:.4e}')
print(f'  GW f0        = {true_gw["f0"]:.8f} Hz')
print(f'  GW fdot      = {true_gw["fdot"]:.4e} Hz/s')
print(f'  GW phi0      = {true_gw["phi0"]:.4f} rad')
print(f'  GW iota      = {true_gw["iota"]:.4f} rad  ({true_iangle_deg:.2f} deg)')
print(f'  EM q (M2/M1) = {true_q:.4f}')

## 2. Build GW likelihood using GBGPU

We inject the true signal via `GBGPU.inject_signal()` to create noise-free data in the A and E TDI channels, then precompute the PSD once. The per-evaluation cost is just one `inject_signal()` call + an inner product (~25 ms).

Sampled GW parameters: `[amp, f0, fdot, phi0, iota]`  
Fixed from catalogue: `fddot=0, psi, lam, beta`

In [ ]:
def make_gw_likelihood(true_params, T=0.25 * 31557600.0, dt=10.0):
    """Create a GW log-likelihood closure using GBGPU.inject_signal()."""
    gb = GBGPU()

    inj = [true_params[k] for k in
           ['amp', 'f0', 'fdot', 'fddot', 'phi0', 'iota', 'psi', 'lam', 'beta']]
    A_data, E_data = gb.inject_signal(*inj, T=T, dt=dt)

    df = 1.0 / T
    f = np.arange(len(A_data)) * df

    # Precompute PSD (expensive — do it once)
    mask = f > 1e-5
    S_A = A1TDISens.get_Sn(f[mask])
    S_E = E1TDISens.get_Sn(f[mask])

    d_A = A_data[mask]
    d_E = E_data[mask]

    fixed_psi = true_params['psi']
    fixed_lam = true_params['lam']
    fixed_beta = true_params['beta']

    print(f'GW likelihood ready: T={T/31557600:.2f} yr, '
          f'nfreq={mask.sum()}, df={df:.4e} Hz')

    def gw_log_likelihood(theta_gw):
        """theta_gw = [amp, f0, fdot, phi0, iota_rad]"""
        amp, f0, fdot, phi0, iota = theta_gw
        try:
            A_h, E_h = gb.inject_signal(
                amp, f0, fdot, 0.0, phi0, iota,
                fixed_psi, fixed_lam, fixed_beta,
                T=T, dt=dt,
            )
        except Exception:
            return -1e100

        diff_A = d_A - A_h[mask]
        diff_E = d_E - E_h[mask]

        chi2 = 4.0 * df * np.sum(
            np.abs(diff_A)**2 / S_A + np.abs(diff_E)**2 / S_E
        ).real

        ll = -0.5 * chi2
        return ll if np.isfinite(ll) else -1e100

    return gw_log_likelihood


print('Building GW likelihood (injecting signal + precomputing PSD)...')
t0 = time.time()
gw_ll = make_gw_likelihood(true_gw)
print(f'  done in {time.time()-t0:.1f}s')

# Sanity check
ll_true = gw_ll([
    true_gw['amp'], true_gw['f0'], true_gw['fdot'],
    true_gw['phi0'], true_gw['iota'],
])
print(f'  log L(true) = {ll_true:.4f}  (should be ~0)')

## 3. Build EM model + synthetic data

In [ ]:
MODEL_TEMPLATE = '/fred/oz480/mcoughli/lcurve/lcurve-rs/test_data/test_model.dat'

em_model = lcurve_rs.Model(MODEL_TEMPLATE)
em_model.set_param('q', true_q)
em_model.set_param('iangle', true_iangle_deg)

# Generate synthetic EM data
ntimes = 100
times = np.linspace(-0.25, 0.75, ntimes)
result = em_model.light_curve(times=times)
flux = np.array(result.flux)

rng = np.random.default_rng(42)
snr_em = 50
ferr = flux.mean() / snr_em
flux_obs = flux + rng.normal(0, ferr, ntimes)

tmpdir = tempfile.mkdtemp()
data_path = os.path.join(tmpdir, 'synthetic.dat')
with open(data_path, 'w') as f:
    for i in range(ntimes):
        f.write(f'{times[i]:.12e} 0.001 {flux_obs[i]:.10e} {ferr:.10e} 1 1\n')

print(f'EM data: {ntimes} points, SNR~{snr_em}')

## 4. Parameter mapping

Joint parameter vector (6 params):
```
[q, iangle_deg, amp, f0, fdot, phi0]
 0      1        2    3    4     5
```

- **EM** uses: `q` (idx 0), `iangle` (idx 1) — no transforms
- **GW** uses: `amp` (idx 2), `f0` (idx 3), `fdot` (idx 4), `phi0` (idx 5), `iota` (idx 1 → radians)
- **Shared**: inclination (EM degrees, GW radians)

In [ ]:
mapping = ParameterMapping(
    joint_names=['q', 'iangle_deg', 'amp', 'f0', 'fdot', 'phi0'],
    em_param_names=['q', 'iangle'],
    gw_param_names=['amp', 'f0', 'fdot', 'phi0', 'iota'],
    em_indices=[0, 1],
    gw_indices=[2, 3, 4, 5, 1],
    em_transforms={},
    gw_transforms={1: lambda x: np.radians(x)},
)

## 5. Priors + JointFitter setup

In [ ]:
q_lo = max(0.01, true_q - 0.1)
q_hi = min(1.0, true_q + 0.1)
i_lo = max(0.0, true_iangle_deg - 10.0)
i_hi = min(90.0, true_iangle_deg + 10.0)
amp_lo = true_gw['amp'] / 3.0
amp_hi = true_gw['amp'] * 3.0
f0_lo = true_gw['f0'] - 1e-7
f0_hi = true_gw['f0'] + 1e-7
fdot_lo = true_gw['fdot'] - 5e-17
fdot_hi = true_gw['fdot'] + 5e-17

priors = ProbDistContainer({
    0: uniform_dist(q_lo, q_hi),
    1: uniform_dist(i_lo, i_hi),
    2: uniform_dist(amp_lo, amp_hi),
    3: uniform_dist(f0_lo, f0_hi),
    4: uniform_dist(fdot_lo, fdot_hi),
    5: uniform_dist(0.0, 2 * np.pi),
})

em_fit_model = em_model.copy()
em_fit_model.set_pparam('q', vary=True, range=0.1)
em_fit_model.set_pparam('iangle', vary=True, range=10.0)

jf = JointFitter(
    em_fit_model, data_path, gw_ll, mapping,
    priors=priors, em_scale=True,
)

# Sanity check
theta_true = [true_q, true_iangle_deg, true_gw['amp'], true_gw['f0'],
              true_gw['fdot'], true_gw['phi0']]
ll_joint = jf.log_likelihood(np.array(theta_true))
print(f'Joint log L(true) = {ll_joint:.2f}')
print(f'  EM component:  {jf.log_likelihood_em(np.array(theta_true)):.2f}')
print(f'  GW component:  {jf.log_likelihood_gw(np.array(theta_true)):.2f}')

## 6. Run MCMC

In [ ]:
NWALKERS = 16
NSTEPS = 300
BURN = 100

print(f'Running eryn MCMC: {NWALKERS} walkers, {NSTEPS} steps, {BURN} burn-in')

t0 = time.time()
result = jf.run_eryn(
    nwalkers=NWALKERS,
    nsteps=NSTEPS,
    burn=BURN,
    ntemps=1,
    vectorize=False,
)
elapsed = time.time() - t0
print(f'\nMCMC completed in {elapsed:.0f}s ({elapsed/60:.1f} min)')

## 7. Parameter recovery

In [ ]:
true_vals = {
    'q': true_q,
    'iangle_deg': true_iangle_deg,
    'amp': true_gw['amp'],
    'f0': true_gw['f0'],
    'fdot': true_gw['fdot'],
    'phi0': true_gw['phi0'],
}

med = result.median
print(f'{"Param":>12s}  {"True":>14s}  {"Median":>14s}  {"Std":>12s}  {"Offset":>8s}')
print('-' * 70)

all_pass = True
for pname in result.param_names:
    true_val = true_vals[pname]
    pi = list(result.param_names).index(pname)
    std = np.std(result.samples[:, pi])
    offset = abs(med[pname] - true_val) / std if std > 0 else np.inf
    status = 'ok' if offset < 3 else 'FAIL'
    if offset >= 3:
        all_pass = False
    print(f'{pname:>12s}  {true_val:14.6e}  {med[pname]:14.6e}  {std:12.4e}  {offset:5.1f} sig  {status}')

print(f'\nOverall: {"ALL RECOVERED within 3 sigma" if all_pass else "SOME FAILED"}')

## 8. EM-only comparison

Compare posterior widths with EM-only fitting to show the benefit of including GW data.

In [ ]:
em_only_model = em_model.copy()
em_only_model.set_pparam('q', vary=True, range=0.1)
em_only_model.set_pparam('iangle', vary=True, range=10.0)

fitter = Fitter(
    em_only_model, data_path, scale=True,
    priors={
        'q': Prior.uniform('q', q_lo, q_hi),
        'iangle': Prior.uniform('iangle', i_lo, i_hi),
    },
)
em_result = fitter.run_eryn(nwalkers=NWALKERS, nsteps=NSTEPS, burn=BURN)

print(f'\n{"Param":>12s}  {"EM-only std":>12s}  {"Joint std":>12s}  {"Improvement":>12s}')
print('-' * 55)
for pname in ['q', 'iangle_deg']:
    ji = list(result.param_names).index(pname)
    joint_std = np.std(result.samples[:, ji])
    em_pname = 'iangle' if pname == 'iangle_deg' else pname
    ei = list(em_result.param_names).index(em_pname)
    em_std = np.std(em_result.samples[:, ei])
    improvement = em_std / joint_std if joint_std > 0 else np.inf
    print(f'{pname:>12s}  {em_std:12.4e}  {joint_std:12.4e}  {improvement:10.2f}x')

## 9. Corner plot

In [ ]:
import corner
import matplotlib.pyplot as plt

truths = [true_vals[p] for p in result.param_names]
fig = corner.corner(
    result.samples,
    labels=list(result.param_names),
    truths=truths,
    show_titles=True,
    title_fmt='.4e',
    quantiles=[0.16, 0.5, 0.84],
)
fig.suptitle('ZTFJ1539 \u2014 Joint GW+EM (real GBGPU)', fontsize=14, y=1.02)
plt.show()

In [ ]:
# Cleanup temp data
os.remove(data_path)
os.rmdir(tmpdir)